In [ ]:
# Practical 8: Stock Market Prediction using LSTM
# Code by Parthiv Abhani

# Install yfinance
!pip -q install yfinance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# ---------------------------------------------------------
# 1. Download Historical NASDAQ Stock Data
# ---------------------------------------------------------

stock = "AAPL"

data = yf.download(
    stock,
    start="2015-01-01",
    end="2025-01-01",
    auto_adjust=True
)

# Use Closing Price
prices = data["Close"].values.reshape(-1, 1)

print("Total data points:", len(prices))
print(data.head())

# ---------------------------------------------------------
# 2. Normalize Stock Prices
# ---------------------------------------------------------

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_prices = scaler.fit_transform(prices)

# ---------------------------------------------------------
# 3. Create Sequential Data
# ---------------------------------------------------------

sequence_length = 60

X = []
y = []

for i in range(sequence_length, len(scaled_prices)):
    X.append(scaled_prices[i-sequence_length:i, 0])
    y.append(scaled_prices[i, 0])

X = np.array(X)
y = np.array(y)

# Reshape for LSTM
X = X.reshape(X.shape[0], X.shape[1], 1)

# ---------------------------------------------------------
# 4. Split Data into Training and Testing
# ---------------------------------------------------------

train_size = int(len(X) * 0.8)

X_train = X[:train_size]
X_test = X[train_size:]

y_train = y[:train_size]
y_test = y[train_size:]

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# ---------------------------------------------------------
# 5. Build LSTM Model
# ---------------------------------------------------------

model = tf.keras.Sequential([
    tf.keras.layers.LSTM(
        50,
        return_sequences=True,
        input_shape=(X_train.shape[1], 1)
    ),

    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.LSTM(50),

    tf.keras.layers.Dropout(0.2),

    tf.keras.layers.Dense(25, activation="relu"),

    tf.keras.layers.Dense(1)
])

# ---------------------------------------------------------
# 6. Compile Model
# ---------------------------------------------------------

model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

model.summary()

# ---------------------------------------------------------
# 7. Train Model
# ---------------------------------------------------------

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# ---------------------------------------------------------
# 8. Make Predictions
# ---------------------------------------------------------

predictions = model.predict(X_test)

# Convert predictions back to original price scale
predicted_prices = scaler.inverse_transform(predictions)

actual_prices = scaler.inverse_transform(
    y_test.reshape(-1, 1)
)

# ---------------------------------------------------------
# 9. Calculate RMSE
# ---------------------------------------------------------

rmse = np.sqrt(
    mean_squared_error(actual_prices, predicted_prices)
)

print("\nRMSE:", round(rmse, 2))

# ---------------------------------------------------------
# 10. Visualize Actual vs Predicted Prices
# ---------------------------------------------------------

plt.figure(figsize=(14, 6))

plt.plot(
    actual_prices,
    label="Actual Price"
)

plt.plot(
    predicted_prices,
    label="Predicted Price"
)

plt.title("AAPL Stock Price: Actual vs Predicted")
plt.xlabel("Time")
plt.ylabel("Stock Price (USD)")
plt.legend()

plt.show()

# ---------------------------------------------------------
# 11. Display Sample Predictions
# ---------------------------------------------------------

results = pd.DataFrame({
    "Actual Price": actual_prices.flatten(),
    "Predicted Price": predicted_prices.flatten()
})

print("\nSample Predictions:")
print(results.head(10))